# သင်ခန်းစာ ၁၆ - Microsoft Foundry ဖြင့် စွမ်းဆောင်နိုင်သော Agent များဖြန့်ချိခြင်း

ဒီနိုတ်ဘွတ်တွင် သင်သည် မီတာစနစ်ပြုလုပ်ပြီး အသုံးပြုနိုင်သော ဖောက်သည်ထောက်ခံဆိုင်ရာ Agent တစ်ခုကို **Contoso** ဟု အမည်ပေးထားသော ကုမ္ပဏီအတွက် တည်ဆောက်ပါမည်။ အစှေ့ကနေ အရင်သင်ခန်းစာတွေကဲ့သို့ Agent ၏ ရုပ်အလှပင်း loop မဟုတ်ပေမယ့် — ဒါဟာ အကြောင်းရင်းဖြစ်ပြီး စက်မောင်းကောင်းစွာ လည်ပတ်ရန် Agent ကိုလုံခြုံစေသည့် *အလိုက်အလျောက်* အရာအားလုံးဖြစ်ပါတယ်။

1. **စက်ပစ္စည်းခေါ်ယူခြင်း** — အမှာစာရှာဖွေရေးနှင့် လက်မှတ်ဖန်တီးခြင်း။
2. **RAG** — သိပ္ပံအခြေခံထားသည့် မူဝါဒဖြေကြားချက်။
3. **မှတ်ဉာဏ်** — အကြိမ်ကြိမ် စကားပြောဆိုရာတွင် ဖောက်သည်အား မှတ်မိခြင်း။
4. **မော်ဒယ်လမ်းညွှန်ခြင်း** — ရိုးရှင်းသော တောင်းဆိုချက်များကို မော်ဒယ်သေးသိုငယ်သို့၊ ပြင်းထန်သောတောင်းဆိုချက်များကို မော်ဒယ်ကြီးသို့ ပို့ပေးခြင်း။
5. **တုံ့ပြန်ချက်သိမ်းဆည်းခြင်း** — မော်ဒယ်ခေါ်မှုမရှိဘဲ အကြိမ်ကြိမ် မေးခွန်းများကို ဖြေကြားပေးခြင်း။
6. **လူ့အတည်ပြုချက်** — အတိုးငွေ ထက်ဝက်ပိုပါက လက်မှတ်ထိုးရန်ရပ်ဆိုင်းခြင်း။
7. **အကဲဖြတ်ခြင်းကမ်းလှမ်းချက်** — မကောင်းသောထုတ်လွှင့်မှုများကို တားဆီးသည့် အော့ဖ်လိုင်း စမ်းသပ်မှု။
8. **ကြည့်ရှုနိုင်မှု** — တောင်းဆိုမှုတစ်ခုချင်းစီအတွင်း OpenTelemetry အသုံးပြု၍ စနစ်သွားလာမှုခြေရာခံခြင်း။

အပိုင်းတိုင်းမှာ ကိုယ်ပိုင်ဖြစ်ပြီး လည်ပတ်နိုင်သည်။ တစ်ကြောင်းချင်းဖတ်ပါ — ထုတ်လုပ်မှုအခြေခံပစ္စည်းများသည် ရည်ရွယ်စွာ သေးငယ်စွာထားရှိပါသည်။


## တပ်ဆင်မှု

ဤ notebook ကို မလုပ်ဆောင်မီ သေချာစေရန်။

1. **Microsoft Foundry ပရောဂျက်တစ်ခု** အတွက် ပြင်ဆင်ထားသော chat မော်ဒယ် (ဥပမာ `gpt-4.1-mini`) ပါရှိပါသည်။
2. **Azure CLI ဖြင့် မှတ်ပုံတင်ပြီးသား** — terminal တွင် `az login` ဟူ၍ အမိန့်ကို run ပြုလုပ်ပါ။
3. **လိုအပ်သော ပတ်ဝန်းကျင် အပြောင်းအလဲများကို သတ်မှတ်ထားပါ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — သင့် Microsoft Foundry ပရောဂျက်၏ endpoint။
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — သင့် တပ်ဆင်ထားသော မော်ဒယ်အမည်။

RAG အပိုင်းတွင် `AZURE_SEARCH_SERVICE_ENDPOINT` နှင့် `AZURE_SEARCH_API_KEY` ကို သတ်မှတ်ထားလျှင် **Azure AI Search** ကိုအသုံးပြုပြီး၊ Search resource မပါဘဲ notebook ကို ပြုလုပ်နိုင်ရန်အတွက် in-memory search ဖြစ်စေရန် fallback လုပ်ပြီးဖြစ်သည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. ကိရိယာများ

ထုတ်လုပ်မှုကိရိယာများသည် အမှန်တကယ် စနစ်များဆန့်ကျင်၍ အလုပ်လုပ်သည်။ ဒီမှာ ကျွန်တော်တို့ plain Python functions တွေနဲ့ order database နဲ့ ticketing system ကို စိစစ်ပြလိုက်ပါတယ်။ `@tool` decorator က agent ကို သူတို့ကို ဖော်ပြပေးတယ်။

`issue_refund` က `approval_mode="always_require"` ကို သတ်မှတ်ထားတာကို သတိပြုပါ၊ အဲ့ဒါက အနိမ့်ဆုံးနဲ့ အထက်ဆုံး ငွေပြန်အမ်းမှုတွေမှာ လူတစ်ယောက်ရဲ့ အတည်ပြုချက် လိုအပ်မှုကို ပြသတဲ့ primitive ဖြစ်ပြီး ကနဦးမှာအသုံးပြုပါတယ်။


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — မူဝါဒ သိမြင်ချက်အခြေခံ

မူဝါဒဆိုင်ရာမေးခွန်းများ ("သင့်ရဲ့ ပြန်လည်ပေးပို့မှု အချိန်ကာလ ဘယ်လောက်လဲ?") တွေကို မော်ဒယ်ရဲ့ မှတ်ဉာဏ်မှ မဟုတ်ဘဲ အာဏာရှိ သရုပ်ပြမှုမှဖြေရှင်းသင့်သည်။ ကျွန်ုပ်တို့သည် ဂုဏ်ပြုလျှောက်ထားသော အသေးစား သိမြင်ချက်အခြေခံကို ရှာဖွေရန် ကိရိယာအဖြစ် ထည့်သွင်းသည်။

ထုတ်လုပ်မှုတွင် ဤသည်မှာ **Azure AI Search** ဖြစ်သည်။ ဒီနေရာတွင် ကျွန်ုပ်တို့သည် notebook ကို မည်သည့်နေရာတွင်မဆို လည်ပတ်နိုင်ရန် စကားလုံးဖြင့်ရှာဖွေရန် စနစ်ဖြစ်သည့် ညှိနှိုင်းချက်ကို ထည့်သွင်းပေးထားပြီး ပတ်ဝန်းကျင် အပြောင်းအလဲအချက်များ ရှိပါက အလိုအလျောက် Azure AI Search သို့ ပြောင်းလဲသည်။


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. မှတ်ဉာဏ်

ဘယ်သူနဲ့ ပြောဆိုနေတယ်ဆိုတာ မေ့နေတဲ့ အထောက်အပံ့ agent တစ်ယောက်ကတော့ မကောင်းတဲ့ အထောက်အပံ့ agent ဖြစ်ပါတယ်။ ကျွန်ုပ်တို့သည် ဝယ်သူတစ်ဦးချင်းစီအတွက် အသေးစား ပရိုဖိုင် စတိုးတစ်ခုကို ထိန်းသိမ်းထားပြီး agent ၏ညွှန်ကြားချက်များထဲသို့ အကျဉ်းချုပ်သေးသောအချက်အလက်တစ်ချက်ကို ထည့်သွင်းပေးသည်။ ထုတ်လုပ်မှုတွင် ၎င်းသည် မှတ်ဉာဏ်ဝန်ဆောင်မှုတစ်ခုဖြစ်သည် (သင်ခန်းစာ ১৩ ကိုကြည့်ပါ)၊ ဒီနေရာမှာ dict သည် ပုံစံကို တွေ့နိုင်စေသည်။


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 နှင့် 5။ မော်ဒယ်လမ်းကြောင်း နှင့် တုံ့ပြန်မှုသိုလှောင်မှု

တစ်ခုတည်းသော မေးခွန်းအတွက် သုံးစွဲမှုကုန်ကျစရိတ်ကို ထိန်းချုပ်ရာတွင် ညှိနှိုင်းထားသောအချက်နှစ်ခု

- **လမ်းကြောင်းသတ်မှတ်ခြင်း**: သက်သာသော heuristic classifier က မေးခွန်းတစ်ခုသည် သေးငယ်သောမောဒယ်တစ်ခု သို့မဟုတ် ကြီးမားသောမော်ဒယ်တစ်ခုလိုအပ်သည်ကိုဆုံးဖြတ်သည်။
- **သိုလှောင်မှု**: ပုံမှန်ပြန်လည်မေးခွန်းများကို မော်ဒယ်ခေါ်ခြင်းမရှိဘဲ မျက်နှာစာမှ တိုက်ရိုက်တုံ့ပြန်သည်။

ဒီနေရာမှာ classifier ကို ရိုးရှင်းအောင် ပြုလုပ်ထားသည်။ ထုတ်လုပ်မှုတွင် သင့်တော်မှုကို ရှုမှတ်ပြီး Foundry ၏ Model Router ဖြင့် အစားထိုးနိုင်သည်။


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. အေးဂျင့်၊ လူ့အတည်ပြုမှုနှင့် ဆောင်ရွက်မှုကြည့်ရှုနိုင်မှု

ယခုတွင် အထက်ဖော်ပြခဲ့သည့်ကိရိယာများမှ အေးဂျင့်ကို စုပေါင်းပြီး အလွှာတစ်ခုစီကို OpenTelemetry span ဖြင့် ထူထောင်သည်။ `handle_support_request` ဖွင့်တင်းချက်သည် ထုတ်လုပ်မှုတောင်းဆိုမှု ကိုင်တွယ်သူဖြစ်သည်- ‌ကက်ရှ် → လမ်းညွှန် → သံသရာ → အလုပ်လုပ် → ကက်ရှ်။

လူ့အတည်ပြုမှုကို framework မှ ကိုင်တွယ်သည်- `issue_refund` သည် `approval_mode="always_require"` ဖြစ်သောကြောင့် အလုပ်လုပ်ကို အနားထားပြီး ကျွန်ုပ်တို့ ရှင်းလင်း သတ်မှတ်သောအတည်ပြုတောင်းဆိုမှုကို မျက်မြင်ရစေသည်။


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## ၇။ အကဲဖြတ်ခြင်း ဂိတ်

သင်ခန်းစာမှ လွှတ်ပေးခြင်း ဂိတ်ဖြစ်သည် - အော့ဖ်လိုင်း စစ်ဆေးမှု စာရင်းတစ်ရပ်သည် ပညာရှင်အား အမှတ်ပေးပြီး၊ ဖြတ်သန်းမှု အကျိုးရှိမှုကန့်သတ်ချက်ကို ကျော်လွန်ပါကသာ ချထားမှုကို ဆက်လက်လုပ်ဆောင်သည်။ ဒီမှာ scorer ဖြစ်တာက စီးရီးကိုယ်တိုင် ထိန်းသိမ်းဖို့ အလွယ်တကူ စကားလုံး ကွက်လပ်ကို စစ်ဆေးခြင်းဖြစ်သည်။ ထုတ်လုပ်မှုမှာ မိတ်ဆက်နေ့လယ် (LLM) ကို တရားသူအဖြစ် သုံးမယ် သို့မဟုတ် မူကြမ်းသုံးသပ်သူ( frameworks evaluator) ကို (သင်ခန်းစာ ၁၀ ကို ကြည့်ပါ) အသုံးပြုပါမယ်။


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ပေါင်းစပ်ခြင်း: အတုဖြန့်ချိမှု

အောက်တွင် ဖော်ပြထားသော ဆဲလ်သည် သင်ကြားချက်တွင် ဖော်ပြထားသည့် အနှံ့အစပတ်လည်ကို ပြသည်- အကဲဖြတ်တံခါးကို ပြေးဆင်းပြီး၊ ဖြတ်ကျော်ပါက "deploy" ကိုသာ ပြုလုပ်သည်။ ဒါက သင် Foundry Agent Service သို့ agent ဗားရှင်းတစ်ခုကို မြှင့်တင်မည်မီ CI တွင် ပြေးဆင်းမည့် ပုံစံဖြစ်သည်။


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## အကျဉ်းချုပ်

သင်သည် ကိုယ်ပိုင်လုပ်ငန်းဆောင်တာများအားလုံး သိမ်းဆည်းထားသည့် ထုတ်လုပ်မှုအသုံးပြုနိုင်သော ဖောက်သည်ထောက်ခံရေး ကိုယ်စားလှယ်တစ်ဦးကို စုစည်းထားသည်။

- **ကိရိယာများ၊ RAG နှင့် မှတ်ဉာဏ်** က ကိုယ်စားလှယ်အား စွမ်းဆောင်ရည်နှင့် အကြောင်းအရာပေးသည်။
- **မော်ဒယ်လမ်းကြောင်းချိတ်ဆက်မှုနှင့် မှတ်ဉာဏ်သိမ်းဆည်းမှု** သည် စောင့်ဆိုင်းချိန်နှင့် ကုန်ကျစရိတ်ကို ထိန်းချုပ်ထားသည်။
- **လူ့အတည်ပြုချက်** သည် ကြီးမားသောပြန်အမ်းငွေကဲ့သို့အန္တရာယ်မြင့်ရာထူးများကိုကာကွယ်သည်။
- **အကဲဖြတ်ခြင်းတံခါး** သည် သေချာမကောင်းသော ထုတ်ပေးမှုများကို ပိတ်ဆို့သည်။
- **အကွက်ကောက်တမ်း** သည် တောင်းဆိုမှုတိုင်းကို တွေ့မြင်နိုင်စေသည်။

### စိန်ခေါ်မှု

ဒီကိုယ်စားလှယ်ကို ပိုမိုတိုးချဲ့ပါ:

1. **အမျိုးမျိုးသော မော်ဒယ်များကို ထောက်ပံ့ရန်** — တတိယ "သိကောင်းစရာ"အဆင့်ကို ထည့်သွင်းပြီး တိုးတက်မှု/တုံ့ပြန်မှုများကို အဲဒီမှာ သွားရောက်ရန် လမ်းကြောင်းမျဉ်းချပြီး။
2. **အကဲဖြတ်တံခါးများ ထည့်သွင်းရန်** — `TEST_CASES` ကို ပြန်အမ်းငွေခွင့်ပြုမှု လမ်းကြောင်းများ ပါဝင်စေရန်ချဲ့ထွင်ပြီး တံခါးက သက်သာမှုများကို ဖမ်းမိနိုင်ကြောင်း သေချာစေရန်။
3. **ကုန်ကျစရိတ်သိရှိသော လမ်းကြောင်းချိတ်ဆက်မှု ထည့်ရန်** — တောင်းဆိုမှုတစ်ခုစီပေါ် အခြေခံပြီး သတ်မှတ်ထားသော ကုန်ကျစရိတ် (သေးငယ်၊ ကြီးမား၊ မှတ်ဉာဏ်မှ) ကို သတင်းအချက်အလက် စုဆောင်းကာ အမျိုးမျိုးသော မေးခွန်းများ ဘက်ချ်ပြီးနောက် ကုန်ကျစရိတ်အစီရင်ခံစာ ထုတ်ပေးရန်။

နောက်တစ်သင်ခန်းစာတွင် သင်သည် ဆန့်ကျင်ဘက်ခရီးကိုယူပြီး ကိုယ်ပိုင်ကွန်ပျူတာ၌ Microsoft Foundry Local နှင့် Qwen ဖြင့် ကိုယ်စားလှယ်တစ်ဦးကို ပြေးဆွဲပါမည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
